# perchv2-pytorch quickstart (native PyTorch / `timm`-based backbone)

This notebook walks through all three usage modes for the **native PyTorch backbone** (`Perch2Backbone`/`Perch2Classifier`/`Perch2Embedder`), in order:

1. **Frozen features** — `Perch2Embedder`, no training at all.
2. **Linear probing** — backbone frozen, only a new head is trained.
3. **Full fine-tuning** — backbone unfrozen, gradients flow through the whole EfficientNet-B3 stack.

Everything below runs against a synthetic 3-class sine-tone dataset with no download and no real audio required, so you can run this notebook top to bottom with nothing but `pip install -e .` first.

**The last section spells out exactly what to change to point this at your own data and your own converted Perch v2 weights.**



In [16]:
import sys
import warnings
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download

# This notebook lives in legacy/, one level below the repo root.
# "legacy" is NOT an installed package (unlike perchv2_pytorch) -- it
# needs the repo root added to sys.path to be importable at all.
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "perchv2_pytorch").exists() is False else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "examples"))

from legacy import PerchFrontend, Perch2Classifier, Perch2Embedder
from toy_dataset import ToySineDataset

warnings.filterwarnings("ignore", message="CUDA initialization.*", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Downloads once, then caches locally -- repeated runs won't re-download.
WEIGHTS_PATH = hf_hub_download(
    repo_id="bghani/perch2-pytorch-weights",
    filename="perch_v2_backbone_timm.pt",
    cache_dir=str(REPO_ROOT / "legacy" / "weights"),
)


Using device: cpu


## Section 1 — Frozen features

`Perch2Embedder` wraps `PerchFrontend` (log-mel frontend) + `Perch2Backbone` (EfficientNet-B3) with no classification head at all. This is the mode to use if you just want fixed embeddings to feed into your own downstream classifier, clustering, or search index — no training happens here.

Perch v2 expects 5-second mono clips at 32kHz = 160,000 samples per clip.


In [17]:
embedder = Perch2Embedder(weights_path=WEIGHTS_PATH).to(device)
embedder.eval()

# Replace this with real audio -- shape (batch, 160000) at 32kHz.
batch = torch.zeros(4, 160_000).to(device)

with torch.no_grad():
    embeddings = embedder(batch)

print("Embedding shape:", embeddings.shape)  # (4, 1536)


Embedding shape: torch.Size([4, 1536])


## The toy dataset used below

`ToySineDataset` (in `examples/toy_dataset.py`) generates 3 classes of synthetic audio — each class is a distinct sine-tone frequency (500Hz / 1500Hz / 4000Hz) plus a little Gaussian noise, correctly shaped as 5s/32kHz clips. It exists purely to exercise the training loop mechanics without needing real recordings or a download.

**This is not biologically meaningful data.** Don't read anything into the accuracy numbers below beyond "the training loop runs and gradients flow where they're supposed to."


In [18]:
dataset = ToySineDataset(n_per_class=20)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"{len(dataset)} clips, {dataset.num_classes} classes")
waveform, label = dataset[0]
print("Example clip shape:", waveform.shape, "| label:", label)


60 clips, 3 classes
Example clip shape: torch.Size([160000]) | label: 0


## Section 2 — Linear probing

`mode="linear_probe"` freezes every backbone parameter (`requires_grad=False`) and forces the backbone into `.eval()` even while the rest of the model is in `.train()`, so BatchNorm running stats don't drift during head-only training. Only `head.weight` / `head.bias` receive gradients.

This is the fast, low-data-friendly option — a good baseline to establish before deciding whether full fine-tuning is worth the extra cost.

Since `WEIGHTS_PATH` is `None` above, the backbone here is **randomly initialized**, not loaded from Perch v2. A frozen random backbone can't extract useful features, so expect accuracy to plateau near chance level (1/3 for this 3-class toy set) — that's expected, and is itself a useful illustration of why the pretrained weights matter. Point `WEIGHTS_PATH` at a real converted checkpoint to see this mode do something meaningful.


In [19]:
mel = PerchFrontend()
linear_probe_model = Perch2Classifier(
    num_classes=dataset.num_classes,
    mel=mel,
    weights_path=WEIGHTS_PATH,
    mode="linear_probe",
).to(device)

trainable = [n for n, p in linear_probe_model.named_parameters() if p.requires_grad]
print("Trainable parameters:", trainable)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, linear_probe_model.parameters()), lr=1e-3
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
linear_probe_model.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0.0, 0, 0
    for waveforms, labels in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = linear_probe_model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * waveforms.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += waveforms.size(0)

    print(f"epoch {epoch+1}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.2%}")


Trainable parameters: ['head.weight', 'head.bias']
epoch 1/5  loss=0.9288  acc=85.00%
epoch 2/5  loss=0.5973  acc=100.00%
epoch 3/5  loss=0.3893  acc=100.00%
epoch 4/5  loss=0.2535  acc=100.00%
epoch 5/5  loss=0.1754  acc=100.00%


## Section 3 — Full fine-tuning

`mode="finetune"` unfreezes the entire backbone — this is the capability the official ONNX/tflite Perch v2 releases don't have, since those are inference-only formats with no autograd graph. Every one of EfficientNet-B3's ~340 parameter tensors receives gradients here, not just the head.

We use **discriminative learning rates**: a small LR on the pretrained backbone (it's already a decent initialization — don't wreck it with a big early update) and a larger LR on the freshly-initialized head. This is generally worth doing rather than a single global LR.

Each epoch prints `backbone_grad_norm` as a sanity check — it should be nonzero here, confirming gradients are actually reaching the backbone (they'd be absent in `frozen`/`linear_probe` mode).


In [20]:
mel_ft = PerchFrontend()
finetune_model = Perch2Classifier(
    num_classes=dataset.num_classes,
    mel=mel_ft,
    weights_path=WEIGHTS_PATH,
    mode="finetune",
).to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": finetune_model.backbone.parameters(), "lr": 1e-5},
        {"params": finetune_model.head.parameters(), "lr": 1e-3},
    ]
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
finetune_model.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0.0, 0, 0
    for waveforms, labels in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = finetune_model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()

        backbone_grad_norm = sum(
            p.grad.norm().item()
            for p in finetune_model.backbone.parameters()
            if p.grad is not None
        )

        optimizer.step()

        total_loss += loss.item() * waveforms.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += waveforms.size(0)

    print(
        f"epoch {epoch+1}/{EPOCHS}  loss={total_loss/total:.4f}  "
        f"acc={correct/total:.2%}  backbone_grad_norm={backbone_grad_norm:.4f}"
    )


epoch 1/5  loss=0.9596  acc=80.00%  backbone_grad_norm=8.0747
epoch 2/5  loss=0.6581  acc=100.00%  backbone_grad_norm=6.2223
epoch 3/5  loss=0.4505  acc=100.00%  backbone_grad_norm=28.7206
epoch 4/5  loss=0.3203  acc=100.00%  backbone_grad_norm=6.1455
epoch 5/5  loss=0.1977  acc=100.00%  backbone_grad_norm=6.2276


## Using your own data and weights

To point any of the three sections above at something real, change these things:

**1. Get real Perch v2 weights** (optional for frozen/linear-probe exploration, but the whole point of `mode="finetune"`):
```python
WEIGHTS_PATH = REPO_ROOT / "weights" / "perch_v2_backbone_timm.pt"
```
Drop your converted checkpoint at that path (see the repo README's "Getting the weights" section).

**2. Replace `ToySineDataset` with your own `Dataset`.** It needs to return `(waveform, label)` pairs where:
   - `waveform` is a 1D `float32` tensor of shape `(160000,)` — 5 seconds of mono audio at 32kHz. If your audio is a different sample rate or length, resample and pad/trim to this shape before returning it (e.g. with `torchaudio.transforms.Resample` and `torch.nn.functional.pad`/slicing).
   - `label` is an integer class index.

   A minimal real-audio version looks like:
   ```python
   import torchaudio

   class MyAudioDataset(torch.utils.data.Dataset):
       def __init__(self, filepaths, labels):
           self.filepaths = filepaths
           self.labels = labels

       def __len__(self):
           return len(self.filepaths)

       def __getitem__(self, idx):
           waveform, sr = torchaudio.load(self.filepaths[idx])
           waveform = waveform.mean(dim=0)  # mono
           if sr != 32000:
               waveform = torchaudio.functional.resample(waveform, sr, 32000)
           target_len = 160_000
           if waveform.shape[0] < target_len:
               waveform = torch.nn.functional.pad(waveform, (0, target_len - waveform.shape[0]))
           else:
               waveform = waveform[:target_len]
           return waveform, self.labels[idx]
   ```

**3. Update `num_classes`.** In the cells above this comes from `dataset.num_classes` automatically — as long as your custom `Dataset` also exposes that (or you just pass your own integer directly to `Perch2Classifier(num_classes=...)`), nothing else needs to change.

**4. Re-run Sections 2 and 3** with the real dataset and real weights in place. Everything else — the `mode` switch, the freezing/unfreezing logic, the discriminative learning rates — stays exactly as written.
